In [66]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from dig4bio.io import read_raman_file
from sklearn.linear_model import LinearRegression
from dig4bio.constants import FINGERPRINT_COLUMNS, LABEL_COLUMNS
from dig4bio.cv import augmented_cross_validate_sample_folds
from dig4bio.models import create_model_factory, AveragePredictor
from dig4bio.paths import EXPERIMENTS_FOLDER
from dig4bio.utils import create_new_experiment_run
from sklearn.model_selection import cross_validate, GroupKFold
import shutil

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
EXPERIMENT_FOLDER = EXPERIMENTS_FOLDER / "exp_001_baselines"
run_folder = create_new_experiment_run(EXPERIMENT_FOLDER)
shutil.copy2(EXPERIMENT_FOLDER/'notebook.ipynb',run_folder/'notebook.ipynb');

In [68]:
source_df = read_raman_file(level='processed',subfolder='source_grid_fingerprint_linear',name='source_datasets.csv')
source_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
1353,2352.126181,2379.709669,2348.676749,2300.885469,2242.669217,2109.389460,2116.425885,2181.189087,2298.221393,2563.834082,...,2926.786968,2934.735075,2917.265009,2904.378987,4.91288,1.03532,3.302040,0.0,0,tec
1791,0.002019,0.002007,0.001995,0.001982,0.001969,0.001955,0.001940,0.001926,0.001915,0.001916,...,0.000724,0.000724,0.000724,0.000724,16.73440,0.00000,0.022216,1.0,1,timegate
1962,35579.719221,35573.205441,35573.965171,35517.194408,35381.947043,35168.903855,34976.075165,34949.758624,35062.141915,35090.540854,...,4976.477004,5027.317206,5036.078318,4990.753927,0.79631,1.06674,3.090570,0.0,1,tornado
149,3086.610000,3084.850000,3083.090000,3078.120000,3073.150000,3055.815000,3038.480000,3015.820000,2993.160000,2979.260000,...,1154.830000,1155.990000,1139.990000,1123.990000,0.00000,0.75931,0.017697,1.0,2,anton532
1053,3495.888412,3524.637931,3498.775862,3472.913793,3479.472103,3493.635193,3513.620690,3539.913793,3556.849785,3524.660944,...,1306.859649,1294.435294,1293.258824,1292.087719,1.96857,0.73647,0.000000,0.0,4,metrohm
2211,34266.136668,34145.233575,34073.724183,34045.031124,34020.562814,33961.889923,33888.328561,33844.666816,33830.193937,33800.477035,...,5427.651435,5365.264347,5353.578959,5386.490633,10.71080,0.74408,3.480790,0.0,4,tornado
2004,16343.835946,16253.854336,16147.248493,16077.812135,16079.821429,16172.057807,16277.582881,16274.635542,16184.681637,16180.376228,...,2802.983448,2793.360985,2809.754676,2818.236826,0.49945,1.33009,0.003688,0.0,1,tornado
1340,10698.510504,10696.297169,10711.907202,10747.198559,10796.517443,10856.571187,10931.915605,11025.551260,11126.420445,11224.213142,...,2185.336550,2188.181566,2166.826788,2131.210891,0.49945,1.33009,0.003688,0.0,4,mettler
200,3350.890000,3348.215000,3345.540000,3343.040000,3340.540000,3325.060000,3309.580000,3280.425000,3251.270000,3233.180000,...,1583.530000,1584.410000,1569.085000,1553.760000,4.63261,0.69969,0.022235,1.0,3,anton532
984,6247.051502,6248.073276,6164.021552,6079.969828,6099.806867,6144.012876,6164.931034,6160.620690,6151.163090,6114.682403,...,1513.865497,1499.964706,1481.141176,1478.824561,1.04985,0.75302,0.009082,0.0,3,metrohm


In [69]:
transfer_plate_df = read_raman_file(level='processed',subfolder='transfer_grid_fingerprint_linear',name='transfer_plate.csv')
transfer_plate_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
26,9656.8750,9645.956522,9615.521739,9575.80625,9526.43125,9454.111801,9411.550,9452.8,9490.496894,9520.50000,...,1689.801242,1684.832298,1684.68750,1690.93750,1726.540373,1737.9625,sample14,8.608633,0.456045,1.905154
146,9649.5000,9608.378882,9602.167702,9589.20000,9569.20000,9478.552795,9417.675,9455.8,9491.229814,9514.86250,...,1677.621118,1693.149068,1700.65625,1697.53125,1693.906832,1682.2000,sample74,7.080100,1.204923,2.470078
76,9713.8125,9605.298137,9628.279503,9603.25000,9528.25000,9489.596273,9453.275,9411.4,9468.534161,9560.01250,...,1655.440994,1691.465839,1698.29375,1666.41875,1687.807453,1699.9375,sample39,11.328211,0.324832,0.942181
105,9701.8125,9691.925466,9711.180124,9691.90625,9632.53125,9570.167702,9524.650,9523.4,9539.049689,9560.95625,...,1681.472050,1693.273292,1696.23750,1687.48750,1698.888199,1699.4500,sample53,11.887974,0.276526,0.487412
85,9695.0000,9573.658385,9620.242236,9649.51250,9660.76250,9587.372671,9514.225,9468.6,9519.043478,9597.71875,...,1649.068323,1675.776398,1701.21250,1724.96250,1731.062112,1715.1250,sample43,7.820476,0.375500,1.633674
57,9736.9375,9656.099379,9697.093168,9674.51250,9585.76250,9535.093168,9497.800,9482.8,9545.577640,9627.71875,...,1675.291925,1707.590062,1713.10000,1683.10000,1688.310559,1683.5625,sample29,11.409887,0.646382,2.183104
140,9604.3750,9547.745342,9555.198758,9521.94375,9446.31875,9446.844720,9453.300,9450.8,9455.913043,9460.98125,...,1693.273292,1700.105590,1708.56875,1719.19375,1707.080745,1683.9625,sample71,8.085028,1.992767,1.996858
24,8198.5000,8210.198758,8192.186335,8126.34375,8010.71875,8023.981366,8031.700,7984.2,8015.459627,8076.76875,...,1606.372671,1615.689441,1624.22500,1631.72500,1630.422360,1614.2500,sample13,10.624163,0.980752,1.429507
51,9677.5625,9697.975155,9637.726708,9587.40000,9547.40000,9469.478261,9412.100,9419.6,9466.347826,9523.32500,...,1674.496894,1686.919255,1697.76250,1706.51250,1702.329193,1687.5125,sample26,9.048045,1.532648,0.528093
3,9666.0000,9604.366460,9618.031056,9601.72500,9554.22500,9485.981366,9434.300,9431.8,9455.919255,9483.65625,...,1723.832298,1694.639752,1680.95625,1687.83125,1714.260870,1716.5500,sample2,5.782718,1.175902,1.214738


In [70]:
### Linear Sources: Linear regression over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(LinearRegression())

sources_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_lr_errors['devices']))
print('Overall')
display(pd.Series(sources_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-9.814037,-11.112856,-268.714326,-520.361891,-178.036716,-11937.375305,-3.594700,-2182.403212
Na_acetate,-13.009612,-0.868218,-17.183729,-67.974554,-209.363922,-6417.786683,-0.516098,-1015.802629
Mg_SO4,-5.064514,-0.036047,-9.014977,-95.357735,-259.089710,-360.502686,-0.240201,-957.250087


Overall


glucose      -2201.480448
Na_acetate   -1198.156705
Mg_SO4        -277.009218
dtype: float64

In [71]:
### Averages Sources: Analyte averages over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(AveragePredictor())

sources_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_avgs_errors['devices']))
print('Overall')
display(pd.Series(sources_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.007701,-0.008376,-0.004897,-0.004898,0.003036,-0.000517,-0.007763,-0.020530
Na_acetate,-0.027002,-0.022158,-0.053501,0.000620,-0.026815,-0.003065,-0.053188,-0.003543
Mg_SO4,-0.076486,-0.077122,-0.237535,-0.023226,-0.065905,-0.022300,-0.258090,-0.014577


Overall


glucose      -0.003056
Na_acetate   -0.003696
Mg_SO4       -0.020624
dtype: float64

In [72]:
### Linear Target: Linear regression train + test over the target device only ###

model_factory = create_model_factory(LinearRegression())

target_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_lr_errors['devices']))
print('Overall')
display(pd.Series(target_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,0.515546,0.528299,0.882741,-3.019211,0.907405,0.612470,0.517172,0.121881
Na_acetate,-0.505592,0.778544,0.821707,0.482408,0.556746,-2.572058,0.690904,-1.108420
Mg_SO4,0.549873,0.835158,0.847339,0.614719,0.879381,-2.023251,0.694120,-10.158555


Overall


glucose       0.028888
Na_acetate   -0.231723
Mg_SO4       -1.627209
dtype: float64

In [73]:
### Linear Averages: Analyte Averages train + test over the target device only ###

model_factory = create_model_factory(AveragePredictor())

target_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_avgs_errors['devices']))
print('Overall')
display(pd.Series(target_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.034112,-0.073546,-0.008200,-0.035465,-0.053714,-0.018685,-0.021394,-0.008342
Na_acetate,-0.056263,-0.023727,-0.027337,-0.020718,-0.014491,-0.030386,-0.007855,-0.022776
Mg_SO4,-0.036037,-0.067311,-0.020365,-0.039999,-0.043428,-0.017934,-0.026028,-0.034934


Overall


glucose      -0.032043
Na_acetate   -0.008138
Mg_SO4        0.000577
dtype: float64

In [ ]:
model = LinearRegression()

test_df = read_raman_file(name='96_samples',level='processed',subfolder='96samples_grid_fingerprint_linear')

X = transfer_plate_df[FINGERPRINT_COLUMNS]
y = transfer_plate_df[LABEL_COLUMNS]

model.fit(X,y)
test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])

display(test_df)

output_df = (
    test_df.copy()
    .groupby(by='sample',sort=False)
    .mean()
    .reset_index()
)[LABEL_COLUMNS]

output_df.index.name = 'ID'
output_df = output_df.reset_index()
output_df['ID'] +=1
output_df.to_csv(run_folder/f'{EXPERIMENT_FOLDER.name}_{str(run_folder.name[4:])}.csv',index=False)
output_df

# Public Kaggle score of this: -1.23611

C:\Users\J_Bra\AppData\Local\Temp\ipykernel_32320\1611781104.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])


,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
0,9459.1250,9440.062112,9440.683230,9403.02500,9325.52500,9295.440994,9269.600,9239.6,9269.279503,9315.65000,...,1650.819876,1671.316770,1689.71875,1705.34375,1696.142857,1672.2125,sample1,1.461898,0.685178,0.353928
1,9441.6250,9411.422360,9435.645963,9429.93125,9393.05625,9312.714286,9256.075,9274.2,9299.850932,9323.63125,...,1655.919255,1678.900621,1679.10000,1649.10000,1665.136646,1674.9625,sample1,2.351013,0.583927,0.467618
2,8607.1875,8614.714286,8571.857143,8517.23125,8450.35625,8408.726708,8371.900,8339.4,8352.229814,8381.90000,...,1603.565217,1642.695652,1666.07500,1668.57500,1651.987578,1625.0000,sample2,5.581757,1.732026,1.431091
3,8560.0625,8553.944099,8543.385093,8527.58750,8506.33750,8434.900621,8377.950,8374.2,8410.590062,8458.28125,...,1609.695652,1627.086957,1630.55000,1615.55000,1632.074534,1635.8375,sample2,6.211434,1.766747,1.229222
4,9432.6875,9347.453416,9321.987578,9321.55625,9347.18125,9274.819876,9216.275,9224.4,9251.919255,9279.65625,...,1677.118012,1705.068323,1704.33750,1665.58750,1693.024845,1707.9500,sample3,4.254881,0.494294,0.999877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,9424.8750,9449.583851,9415.422360,9380.85000,9345.85000,9276.770186,9215.950,9187.2,9226.993789,9288.96875,...,1638.571429,1652.857143,1652.13125,1631.50625,1642.950311,1645.8625,sample94,3.863245,0.210255,0.701643
188,9286.9375,9219.875776,9218.633540,9176.35000,9091.35000,9072.602484,9057.475,9030.6,9054.521739,9094.01250,...,1625.894410,1648.254658,1663.95625,1670.83125,1652.925466,1624.9625,sample95,2.050828,1.525044,0.744649
189,9425.3750,9358.006211,9348.068323,9301.35000,9216.35000,9203.683230,9197.850,9186.6,9245.931677,9329.59375,...,1638.242236,1669.298137,1678.40000,1658.40000,1670.981366,1669.5000,sample95,1.719697,1.459425,0.542773
190,9433.0625,9367.043478,9297.478261,9233.82500,9176.32500,9134.068323,9090.300,9037.8,9117.720497,9246.71875,...,1609.689441,1651.925466,1668.74375,1651.86875,1650.546584,1638.9375,sample96,4.173476,1.753767,0.483940


,ID,glucose,Na_acetate,Mg_SO4
0,1,1.906455,0.634553,0.410773
1,2,9.145176,0.833975,0.326851
2,3,4.515469,1.227353,1.198945
3,4,0.097181,0.947006,0.940205
4,5,0.501357,1.248227,1.221857
...,...,...,...,...
91,92,0.654263,1.251363,1.186624
92,93,4.352456,0.247024,0.550650
93,94,3.855683,0.297089,0.666311
94,95,1.885262,1.492234,0.643711


In [75]:
comparison = pd.DataFrame(
    (sources_avgs_errors['total'],
    sources_lr_errors['total'],
    target_avgs_errors['total'],
    target_lr_errors['total']),
).T

comparison.columns = ['Averages Sources', 'Linear Sources', 'Averages Target', 'Linear Target']
comparison

,Averages Sources,Linear Sources,Averages Target,Linear Target
glucose,-0.003056,-2201.480448,-0.032043,0.028888
Na_acetate,-0.003696,-1198.156705,-0.008138,-0.231723
Mg_SO4,-0.020624,-277.009218,0.000577,-1.627209


In [76]:
comparison.mean(axis=0)

Averages Sources      -0.009125
Linear Sources     -1225.548790
Averages Target       -0.013202
Linear Target         -0.610015
dtype: float64